# Publications markdown generator for academicpages

Takes a TSV of publications with metadata and converts them for use with [academicpages.github.io](academicpages.github.io). This is an interactive Jupyter notebook ([see more info here](http://jupyter-notebook-beginner-guide.readthedocs.io/en/latest/what_is_jupyter.html)). The core python code is also in `publications.py`. Run either from the `markdown_generator` folder after replacing `publications.tsv` with one containing your data.

TODO: Make this work with BibTex and other databases of citations, rather than Stuart's non-standard TSV format and citation style.


## Data format

The TSV needs to have the following columns: pub_date, title, venue, excerpt, citation, site_url, and paper_url, with a header at the top. 

- `excerpt` and `paper_url` can be blank, but the others must have values. 
- `pub_date` must be formatted as YYYY-MM-DD.
- `url_slug` will be the descriptive part of the .md file and the permalink URL for the page about the paper. The .md file will be `YYYY-MM-DD-[url_slug].md` and the permalink will be `https://[yourdomain]/publications/YYYY-MM-DD-[url_slug]`

This is how the raw file looks (it doesn't look pretty, use a spreadsheet or other program to edit and create).

In [1]:
!cat publications.tsv

pub_date	title	venue	excerpt	citation	url_slug	paper_url	link	Code	github
24-03-2011	Combining GLCM Features and Markov Random Field Model for Colour Textured Image Segmentation	International Conference on Devices and Communications (ICDeCom)	This paper is about the number 1. The number 2 is left for future work.	"J. Mridula; Kundan Kumar; Dipti Patra. (2011). ""Combining GLCM Features and Markov Random Field Model for Colour Textured Image Segmentation."" <i>International Conference on Devices and Communications (ICDeCom)</i>. pp.1-5. IEEE, DOI: 10.1109/ICDECOM.2011.5738494"	GLCM_Segmentation		https://doi.org/10.1109/ICDECOM.2011.5738494		
22-12-2011	Blood microscopic image segmentation using rough sets	International Conference on Image Information Processing		"Subrajeet Mohapatra; Dipti Patra; Kundan Kumar. (2011). ""Blood microscopic image segmentation using rough sets."" <i>International Conference on Image Information Processing</i>. pp.1-6. IEEE, DOI: 10.1109/ICIIP.2011.6108977"	

## Import pandas

We are using the very handy pandas library for dataframes.

In [2]:
import pandas as pd

## Import TSV

Pandas makes this easy with the read_csv function. We are using a TSV, so we specify the separator as a tab, or `\t`.

I found it important to put this data in a tab-separated values format, because there are a lot of commas in this kind of data and comma-separated values can get messed up. However, you can modify the import statement, as pandas also has read_excel(), read_json(), and others.

In [3]:
publications = pd.read_csv("publications.tsv", sep="\t", header=0)
publications


,pub_date,title,venue,excerpt,citation,url_slug,paper_url,link,Code,github
0,24-03-2011,Combining GLCM Features and Markov Random Fiel...,International Conference on Devices and Commun...,This paper is about the number 1. The number 2...,J. Mridula; Kundan Kumar; Dipti Patra. (2011)....,GLCM_Segmentation,NaN,https://doi.org/10.1109/ICDECOM.2011.5738494,NaN,NaN
1,22-12-2011,Blood microscopic image segmentation using rou...,International Conference on Image Information ...,NaN,Subrajeet Mohapatra; Dipti Patra; Kundan Kumar...,Rough_Set_Segmentation,NaN,https://doi.org/10.1109/ICIIP.2011.6108977,NaN,NaN
2,01-03-2012,Unsupervised leukocyte image segmentation usin...,International Scholarly Research Notices,NaN,"Subrajeet Mohapatra ,Dipti Patra ,Kundan Kumar...",Rough_Fuzzy_Clustering,/files/pdf/Publications/2012_RoughSet_Clusteri...,https://doi.org/10.5402/2012/923946,NaN,NaN
3,21-03-2012,Fast leukocyte image segmentation using shadow...,International Journal of Computational Biology...,NaN,"Mohapatra, S., Patra, D., & Kumar, K. (2012). ...",Shadowed_Set_Segmentation,NaN,https://www.inderscienceonline.com/doi/abs/10....,NaN,NaN
4,26-07-2020,Time series-based SHM using PCA with applicati...,Journal of Civil Structural Health Monitoring,NaN,"Kumar, K., Biswas, P. K., & Dhang, N. (2020). ...",PCA_ASCE_SHM,NaN,https://link.springer.com/article/10.1007/s133...,NaN,NaN
5,19-02-2020,Automated retinal vessel segmentation based on...,Advanced Computing and Intelligent Engineering,NaN,"Kumar K., Samal D., Suraj (2020) Automated Ret...",GaborWavelet_RetinalImage_Segmentation,NaN,https://doi.org/10.1007/978-981-15-1081-6_35,NaN,NaN
6,10-02-2020,Blood Vessel Detection Using Modified Multisca...,International Conference on Applied Machine Le...,NaN,"Kumar, K., Mallick, D., and Agarwal, S. (2019)...",MF-FDOG_Retinal_Segmentation,NaN,https://doi.org/10.1109/ICAML48257.2019.00024,NaN,NaN
7,06-05-2019,Damage Diagnosis of Steel Truss Bridges under ...,International Journal of Acoustics and Vibration,NaN,"Kumar, K., Biswas, P. K., & Dhang, N. (2019). ...",SteelTruss_IJAV,NaN,https://doi.org/10.20855/ijav.2019.24.11255,NaN,NaN
8,08-06-2016,Statistical Damage Detection Approach in SHM B...,International Journal of Structural and Civil ...,NaN,"Kumar, K., Biswas, P. K., & Dhang, N. (2016). ...",EPM_Conf_SHM,NaN,https://doi.org/10.18178/IJSCER.5.4.300-307,NaN,NaN
9,14-07-2016,Data Normalization Technique in SHM under Envi...,23rd International Congress on Sound and Vibra...,NaN,"Kumar, K., Biswas, P. K., & Dhang, N. (2016). ...",OneClassSVM_SHM,NaN,https://d1wqtxts1xzle7.cloudfront.net/46464289...,NaN,NaN


## Escape special characters

YAML is very picky about how it takes a valid string, so we are replacing single and double quotes (and ampersands) with their HTML encoded equivilents. This makes them look not so readable in raw format, but they are parsed and rendered nicely.

In [4]:
html_escape_table = {
    "&": "&amp;",
    '"': "&quot;",
    "'": "&apos;"
    }

def html_escape(text):
    """Produce entities within text."""
    return "".join(html_escape_table.get(c,c) for c in text)

## Creating the markdown files

This is where the heavy lifting is done. This loops through all the rows in the TSV dataframe, then starts to concatentate a big string (```md```) that contains the markdown for each type. It does the YAML metadata first, then does the description for the individual page.

In [5]:
import os
for row, item in publications.iterrows():
    
    md_filename = str(item.pub_date) + "-" + item.url_slug + ".md"
    html_filename = str(item.pub_date) + "-" + item.url_slug
    year = item.pub_date[:4]
    
    ## YAML variables
    
    md = "---\ntitle: \""   + item.title + '"\n'
    
    md += """collection: publications"""
    
    md += """\npermalink: /publication/""" + html_filename
    
    if len(str(item.excerpt)) > 5:
        md += "\nexcerpt: '" + html_escape(item.excerpt) + "'"
    
    md += "\ndate: " + str(item.pub_date) 
    
    md += "\nvenue: '" + html_escape(item.venue) + "'"
    
    if len(str(item.paper_url)) > 5:
        md += "\npaperurl: '" + item.paper_url + "'"
    
    md += "\ncitation: '" + html_escape(item.citation) + "'"
    
    md += "\n---"
    
    ## Markdown description for individual page
        
    if len(str(item.excerpt)) > 5:
        md += "\n" + html_escape(item.excerpt) + "\n"
    
    if len(str(item.paper_url)) > 5:
        md += "\n[Download paper here](" + item.paper_url + ")\n" 
        
    md += "\nRecommended citation: " + item.citation
    
    md_filename = os.path.basename(md_filename)
       
    with open("../_publications/" + md_filename, 'w') as f:
        f.write(md)

These files are in the publications directory, one directory below where we're working from.

In [6]:
!ls ../_publications/

01-03-2012-Rough_Fuzzy_Clustering.md
03-04-2021-Perovskite_SolarCells.md
05-05-2014-WebGirderBridge_SHM.md
06-05-2019-SteelTruss_IJAV.md
08-06-2016-EPM_Conf_SHM.md
10-02-2020-MF-FDOG_Retinal_Segmentation.md
14-07-2016-OneClassSVM_SHM.md
16-12-2021-OptimalContro_RL.md
19-02-2020-GaborWavelet_RetinalImage_Segmentation.md
2011-GLCM_Segmentation.md
2011-Rough_Set_Segmentation.md
20-12-2021-Oral_Histopathological_Image_Residual.md
2012-Rough_Fuzzy_Clustering.md
2012-Shadowed_Set_Segmentation.md
2014-WebGirderBridge_SHM.md
2016-EPM_Conf_SHM.md
2016-OneClassSVM_SHM.md
2019-SteelTruss_IJAV.md
2020-GaborWavelet_RetinalImage_Segmentation.md
2020-MF-FDOG_Retinal_Segmentation.md
2020-PCA_ASCE_SHM.md
2021-Animony_SolarCells.md
2021-MeanShift_CamShift.md
2021-OptimalContro_RL.md
2021-ORS_SHM.md
2021-Perovskite_SolarCells.md
2021-Similarity_Check.md
21-03-2012-Shadowed_Set_Segmentation.md
22-12-2011-Rough_Set_Segmentation.md
24-03-2011-GLCM_Segmentation.md
24-05-2021-Animony_SolarCells.md
26-07-2020-

In [7]:
!cat ../_publications/2009-10-01-paper-title-number-1.md

cat: ../_publications/2009-10-01-paper-title-number-1.md: No such file or directory
